# PosEnc Output Comparison

This notebook has two modes:

- **Saved-output summary** from `out/encoded_*.npy` (optional).
- **Canonical probe diagnostics** using fixed demo settings (`seq_len=64`, `dim=64`) so RoPE-style plots match the original interpretation.

Probe diagnostics are computed from encoder implementations directly (not from random saved vectors).
            


In [ ]:
from pathlib import Path
import json
import sys

import numpy as np

try:
    import matplotlib.pyplot as plt
    HAVE_MPL = True
except Exception as exc:
    HAVE_MPL = False
    print('matplotlib unavailable:', exc)
    print('Install notebook deps with: uv sync --extra notebooks')
            


In [ ]:
REPO_CANDIDATES = [
    Path.cwd(),
    Path.cwd().parent,
    Path('/home/jake/Developer/posenc'),
]

REPO_ROOT = next(
    (
        p.resolve()
        for p in REPO_CANDIDATES
        if (p / 'core' / 'types.py').exists() and (p / 'encoders' / '__init__.py').exists()
    ),
    Path('/home/jake/Developer/posenc').resolve(),
)

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('REPO_ROOT:', REPO_ROOT)

RUN_DIR_CANDIDATES = [
    REPO_ROOT / 'out',
    Path('out').resolve(),
    Path('../out').resolve(),
]
RUN_DIR = next((p for p in RUN_DIR_CANDIDATES if p.exists()), RUN_DIR_CANDIDATES[0])
print('RUN_DIR:', RUN_DIR)


def load_artifacts(run_dir: Path):
    meta_path = run_dir / 'metadata.json'
    vectors_path = run_dir / 'vectors.npy'
    if not meta_path.exists() or not vectors_path.exists():
        raise FileNotFoundError(
            f'Missing artifacts in {run_dir}. Expected metadata.json and vectors.npy.'
        )

    metadata = json.loads(meta_path.read_text())
    vectors = np.load(vectors_path)
    encoded = {
        p.stem.replace('encoded_', ''): np.load(p)
        for p in sorted(run_dir.glob('encoded_*.npy'))
    }
    return metadata, vectors, encoded


try:
    metadata, vectors, encoded = load_artifacts(RUN_DIR)
    print('encoders with saved tensors:', list(encoded.keys()))
    print('saved vectors shape:', vectors.shape)
except FileNotFoundError as exc:
    print(exc)
    print('Generate artifacts with:')
    print('uv run python main.py --encoders all --save-dir out --save-encoded')
    metadata = None
    vectors = None
    encoded = {}
            


In [ ]:
from core.positions import build_position_bank, parse_coords, parse_t_values
from core.types import RunConfig
from encoders import encoder_names, resolve_specs

DEMO_SEQ_LEN = 64
DEMO_DIM = 96
DEMO_THETA_BASE = 10000.0
DEMO_SEED = 0
DEMO_COORDS = 'x'

requested_names = list(metadata.get('encoders', [])) if metadata else []
if not requested_names:
    requested_names = list(encoder_names())

# Keep order stable and remove duplicates.
seen = set()
requested_names = [name for name in requested_names if not (name in seen or seen.add(name))]
requested_names = [name for name in requested_names if name != 'ape']

probe_contexts = {}
skipped = {}

for name in requested_names:
    per_encoder_params = {}
    if name == 'spiral':
        per_encoder_params = {'num_directions': 1}

    cfg = RunConfig(
        encoders=(name,),
        dim=DEMO_DIM,
        num_vectors=2,
        seed=DEMO_SEED,
        theta_base=DEMO_THETA_BASE,
        coords_spec=parse_coords(DEMO_COORDS),
        grid_size=DEMO_SEQ_LEN,
        centered_coords=False,
        t_values=parse_t_values([0.0]),
        z_value=0.0,
        position_chunk_size=DEMO_SEQ_LEN,
        save_dir=None,
        save_encoded=False,
        encoder_params={name: per_encoder_params} if per_encoder_params else {},
    )

    spec = resolve_specs([name])[0]
    check = spec.validate_config(cfg)
    if not check.ok:
        skipped[name] = check.rule
        continue

    bank = build_position_bank(
        cfg.coords_spec,
        cfg.grid_size,
        cfg.centered_coords,
        cfg.t_values,
        cfg.z_value,
    )
    cache = spec.precompute(cfg, bank)

    probe_contexts[name] = {
        'cfg': cfg,
        'spec': spec,
        'bank': bank,
        'cache': cache,
    }

print('probe-enabled encoders:', list(probe_contexts.keys()))
if skipped:
    print('skipped encoders (incompatible with demo dim={DEMO_DIM}):')
    for name, rule in skipped.items():
        print(f'  - {name}: {rule}')
            


In [ ]:
if not encoded:
    print('No saved encoded tensors loaded; skipping saved-output norm summary.')
else:
    mean_norm_by_encoder = {}
    for name, tensor in encoded.items():
        norms = np.linalg.norm(tensor, axis=2)
        mean_norm_by_encoder[name] = float(np.mean(norms))

    if HAVE_MPL:
        plt.figure(figsize=(8, 4))
        plt.bar(mean_norm_by_encoder.keys(), mean_norm_by_encoder.values())
        plt.ylabel('mean output norm')
        plt.title('Mean encoded norm by encoder (saved outputs)')
        plt.xticks(rotation=30, ha='right')
        plt.tight_layout()
        plt.show()
    else:
        for name, value in mean_norm_by_encoder.items():
            print(f'{name}: {value:.6f}')
            


In [ ]:
if not probe_contexts:
    print('No compatible encoders for probe positional heatmaps.')
elif not HAVE_MPL:
    print('matplotlib unavailable; skipping probe positional heatmaps.')
else:
    def build_probe(name: str, cache, dim: int) -> np.ndarray:
        if name == 'f-monster' and hasattr(cache, 'axis'):
            num_freq = cache.axis.shape[0]
            probe = np.zeros((1, dim), dtype=np.float64)

            for k in range(num_freq):
                a = cache.axis[k]
                ref = np.array([1.0, 0.0, 0.0], dtype=np.float64)
                if abs(a[0]) > 0.9:
                    ref = np.array([0.0, 1.0, 0.0], dtype=np.float64)

                u = ref - np.dot(ref, a) * a
                norm = np.linalg.norm(u)
                if norm < 1e-12:
                    ref = np.array([0.0, 0.0, 1.0], dtype=np.float64)
                    u = ref - np.dot(ref, a) * a
                    norm = np.linalg.norm(u)
                if norm < 1e-12:
                    raise ValueError('Failed to build orthogonal probe vector for f-monster.')

                u = u / norm
                probe[0, 4 * k : 4 * k + 4] = np.array([0.0, u[0], u[1], u[2]], dtype=np.float64)
            return probe

        probe = np.zeros((1, dim), dtype=np.float64)
        probe[:, 0::2] = 1.0
        return probe

    pair_align_names = {'rope', 'axial', 'spiral'}
    mats = {}

    for name in requested_names:
        if name not in probe_contexts:
            continue

        ctx = probe_contexts[name]
        spec = ctx['spec']
        cache = ctx['cache']
        cfg = ctx['cfg']

        probe = build_probe(name, cache, DEMO_DIM)
        out = spec.apply(probe, cache, cfg.position_chunk_size)
        mat = out[0]  # (seq_len, dim)

        if name in pair_align_names:
            aligned = np.empty_like(mat)
            aligned[:, 0::2] = mat[:, 1::2]
            aligned[:, 1::2] = mat[:, 0::2]
            mat = aligned

        mats[name] = mat[:DEMO_SEQ_LEN, :DEMO_DIM]

    if not mats:
        print('No probe matrices computed.')
    else:
        global_min = min(float(np.min(v)) for v in mats.values())
        global_max = max(float(np.max(v)) for v in mats.values())
        if abs(global_max - global_min) < 1e-12:
            global_max = global_min + 1e-12

        print(f'Probe positional heatmap shared range: [{global_min:.6f}, {global_max:.6f}]')
        if 'f-monster' in mats:
            print('f-monster probe uses per-block [0, u_k] with u_k ⟂ axis_k (pure rotation probe).')

        # Sanity check: rope probe should match canonical RoPE construction for this demo dim.
        if 'rope' in mats:
            pos = np.arange(DEMO_SEQ_LEN, dtype=np.float64)[:, None]
            pair_idx = np.arange(0, DEMO_DIM, 2, dtype=np.float64)
            inv_freq = DEMO_THETA_BASE ** (-pair_idx / DEMO_DIM)
            angles = pos * inv_freq[None, :]

            canonical = np.empty((DEMO_SEQ_LEN, DEMO_DIM), dtype=np.float64)
            canonical[:, 0::2] = np.sin(angles)
            canonical[:, 1::2] = np.cos(angles)

            corr = np.corrcoef(mats['rope'].reshape(-1), canonical.reshape(-1))[0, 1]
            print(f'rope probe sanity corr vs canonical dim={DEMO_DIM}: {corr:.6f}')

        for name in requested_names:
            if name not in mats:
                continue

            view = mats[name]
            fig, ax = plt.subplots(figsize=(12, 4.8), constrained_layout=True)
            im = ax.pcolormesh(view, cmap='viridis', shading='auto', vmin=global_min, vmax=global_max)
            ax.set_title(f'{name}: Probe Position vs Encoding Value (dim={DEMO_DIM}, seq={DEMO_SEQ_LEN})')
            ax.set_xlabel('Embedding dimension')
            ax.set_ylabel('Token position')
            ax.set_xlim((0, DEMO_DIM))
            ax.set_ylim((DEMO_SEQ_LEN, 0))
            cbar = fig.colorbar(im, ax=ax, pad=0.02)
            cbar.set_label('encoding value')
            plt.show()


In [ ]:
def offset_curve(sim: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    seq_len = sim.shape[0]
    offsets = np.arange(-(seq_len - 1), seq_len)
    means = np.array([np.mean(np.diag(sim, k=o)) for o in offsets], dtype=np.float64)
    return offsets, means

similarities = {}
offset_curves = {}

if not probe_contexts:
    print('No compatible encoders for synthetic q/k attention analysis.')
else:
    rng = np.random.default_rng(0)
    q0 = rng.normal(size=(DEMO_DIM,)).astype(np.float64)
    k0 = rng.normal(size=(DEMO_DIM,)).astype(np.float64)
    base = np.stack((q0, k0), axis=0)

    for name in requested_names:
        if name not in probe_contexts:
            continue

        ctx = probe_contexts[name]
        spec = ctx['spec']
        cache = ctx['cache']
        cfg = ctx['cfg']

        out = spec.apply(base, cache, cfg.position_chunk_size)
        q = out[0]
        k = out[1]

        # Match rope_graph style: unscaled dot product with independent q/k probes.
        sim = q @ k.T
        similarities[name] = sim
        offset_curves[name] = offset_curve(sim)

    print('Built synthetic q/k attention analyses for encoders:', list(similarities.keys()))
            


In [ ]:
if not similarities:
    print('No similarity matrices to plot.')
elif not HAVE_MPL:
    print('matplotlib unavailable; skipping similarity heatmaps.')
else:
    global_abs_max = max(float(np.max(np.abs(sim))) for sim in similarities.values())
    if global_abs_max < 1e-12:
        global_abs_max = 1e-12

    print(f'Similarity heatmap shared symmetric range: [-{global_abs_max:.6f}, +{global_abs_max:.6f}]')

    for name in requested_names:
        if name not in similarities:
            continue

        sim = similarities[name]
        fig, ax = plt.subplots(figsize=(7.5, 6.0), constrained_layout=True)
        im = ax.imshow(sim, cmap='coolwarm', aspect='auto', vmin=-global_abs_max, vmax=global_abs_max)
        ax.set_title(f'{name}: Attention Similarity Matrix (dim={DEMO_DIM}, seq={DEMO_SEQ_LEN})')
        ax.set_xlabel('q position')
        ax.set_ylabel('p position')
        cbar = fig.colorbar(im, ax=ax, pad=0.02)
        cbar.set_label('attention similarity')
        plt.show()
            


In [ ]:
if not offset_curves:
    print('No offset curves to plot.')
elif not HAVE_MPL:
    print('Offset-curve summary (first 5 points per encoder):')
    for name, (offsets, values) in offset_curves.items():
        head = ', '.join(f'{int(o)}:{v:.3f}' for o, v in zip(offsets[:5], values[:5]))
        print(f'  {name}: {head}')
else:
    plt.figure(figsize=(10, 5))
    for name in requested_names:
        if name not in offset_curves:
            continue
        offsets, values = offset_curves[name]
        plt.plot(offsets, values, label=name, linewidth=2)

    plt.axvline(0, color='black', linewidth=1, alpha=0.3)
    plt.title('Attention Score vs Positional Offset (synthetic q/k, dim={DEMO_DIM}, seq={DEMO_SEQ_LEN})')
    plt.xlabel('positional offset (p - q)')
    plt.ylabel('mean dot product')
    plt.grid(alpha=0.25)
    plt.legend()
    plt.tight_layout()
    plt.show()
            
